## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-23-13-53-33 +0000,nyt,"They Were Too Old to Enlist, but Not to Fight....",https://www.nytimes.com/2026/11/23/world/europ...
1,2026-11-23-13-52-40 +0000,bbc,Welfare cuts: What's been happening with Pip a...,https://www.bbc.co.uk/news/articles/cj924xvzrr...
2,2026-11-23-13-51-28 +0000,nyt,Is Mexico Stepping Up in the War on Drugs?,https://www.nytimes.com/2026/11/22/world/ameri...
3,2026-11-23-13-48-53 +0000,nyt,Virginia School District Shaken by Sex Abuse C...,https://www.nytimes.com/2026/11/23/us/albemarl...
4,2026-11-23-13-47-16 +0000,nyt,"Paolo Bulgari, Heir of Italian Jewelry Dynasty...",https://www.nytimes.com/2026/11/23/style/paolo...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2513/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
538,trump,15
126,deal,14
209,new,12
235,russian,12
145,california,12


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
111,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...,61
251,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...,54
36,2026-11-23-12-33-02 +0000,wapo,Ukraine says it sank Russian warship off coast...,https://www.washingtonpost.com/world/2026/11/2...,52
168,2026-11-23-00-00-00 +0000,wsj,Incoming Michigan Democratic Sen.-elect Abdul ...,https://www.wsj.com/politics/abdul-el-sayed-is...,52
46,2026-11-23-12-10-26 +0000,chitri,Ukrainian drones hit Russian warship at naval ...,https://www.chicagotribune.com/2026/11/23/ukra...,51


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
111,61,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...
251,54,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
168,52,2026-11-23-00-00-00 +0000,wsj,Incoming Michigan Democratic Sen.-elect Abdul ...,https://www.wsj.com/politics/abdul-el-sayed-is...
198,49,2026-11-22-20-48-02 +0000,chitri,Chuck Hubbard defeats Rep. Virginia Foxx after...,https://www.chicagotribune.com/2026/11/22/chuc...
238,46,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
47,44,2026-11-23-12-03-00 +0000,wsj,Five things to know about California Rep. Robe...,https://www.wsj.com/politics/policy/five-thing...
108,41,2026-11-23-09-30-17 +0000,nypost,Jensen Huang’s lover was in contact with Chine...,https://nypost.com/2026/11/23/us-news/jensen-h...
45,31,2026-11-23-12-12-18 +0000,nypost,Former Pete Hegseth chief of staff calls on wa...,https://nypost.com/2026/11/23/us-news/former-p...
58,31,2026-11-23-11-54-00 +0000,wsj,Texas Pacific Land to Buy EagleRock Land in $3...,https://www.wsj.com/business/energy-oil/texas-...
99,31,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
